# M8 L8: LLM Fundamentals -- Tokens, Prompts, Cost

**Build a tiny BPE tokenizer, mock LLM, cost tracker, and prompt template from scratch.**

Duration: 90 min.  No API keys.

## Plain English (the 12-year-old version)

Every LLM app is **five things**:

1. A **tokenizer** (turns text into numbers; counts tokens for cost)
2. A **prompt template** (the question you ask, with placeholders for variables)
3. A **model call** (the actual AI answering the question)
4. An **output parser** (pulls structured data out of the answer)
5. A **cost + latency tracker** (logs every call so you know what you're spending)

Get those five right and the rest is plumbing. Most teams skip the fifth one -- and then they're surprised by the bill.

## The 5 things to know

1. **Tokens, not characters** -- models see tokens (3-4 chars each in English). Token count drives cost and latency.
2. **BPE is how tokenizers work** -- merge the most-frequent adjacent byte pair, repeat.
3. **Cost math** -- at gpt-4o-mini, a 1K-in + 500-out call is $0.0005; 1M calls/month = $500.
4. **Latency math** -- TTFT 100-500ms + 20-50ms per output token. Streaming is required.
5. **Always start with the smallest model that meets quality** -- the cost difference is 10-100x.

In [1]:
# === Standard setup ===
import sys, importlib.util
from pathlib import Path

def find_root(p):
    while p != p.parent:
        if (p / 'lessons').exists() and (p / 'SYLLABUS.md').exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

_p = ROOT / 'lessons' / 'm08-llm-fundamentals' / 'L8-llm-fundamentals.py'
_spec = importlib.util.spec_from_file_location('L8', _p)
_L8 = importlib.util.module_from_spec(_spec)
sys.modules['L8'] = _L8
_spec.loader.exec_module(_L8)

print(f'Setup OK. Lesson module: {_L8.LESSON_NUMBER}')

Setup OK. Lesson module: M8.L8


In [2]:
# Run the 12 unit tests inline
import inspect
tests = [(n, fn) for n, fn in inspect.getmembers(_L8, inspect.isfunction) if n.startswith('test_')]
print(f'Running {len(tests)} tests...\n')
passed, failed = 0, 0
for name, fn in tests:
    try:
        fn()
        print(f'  PASS  {name}')
        passed += 1
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
        failed += 1
print(f'\n{passed}/{len(tests)} tests pass, {failed} fail')
assert failed == 0, f'{failed} test(s) failed -- fix before continuing'

Running 24 tests...

  PASS  test_bpe_compresses_repeats
  PASS  test_bpe_handles_unknown
  PASS  test_bpe_roundtrip
  PASS  test_bpe_tokenizer_handles_empty_string
  PASS  test_bpe_tokenizer_roundtrip_preserves_chars
  PASS  test_cost_tracker_aggregates
  PASS  test_cost_tracker_aggregates_consistent
  PASS  test_cost_tracker_p95
  PASS  test_cost_tracker_p95_latency
  PASS  test_llm_client_increments_call_count
  PASS  test_llm_client_logs_cost
  PASS  test_llm_math_invariants
  PASS  test_mock_llm_deterministic
  PASS  test_mock_llm_deterministic_per_prompt
  PASS  test_parse_json_plain
  PASS  test_parse_json_strips_markdown_fence
  PASS  test_parse_json_strips_plain_fence
  PASS  test_parse_json_with_fence
  PASS  test_prompt_template_missing_var_raises
  PASS  test_prompt_template_renders
  PASS  test_prompt_template_renders_all_variables
  PASS  test_run_demo_seed_42_default_seed
  PASS  test_run_demo_seed_42_is_deterministic
  PASS  test_run_demo_seed_42_uses_all_questions

24/

In [3]:
# === Section 1: train a BPE tokenizer ===
tok = _L8.BPETokenizer()
tok.fit(_L8.SAMPLE_CORPUS, vocab_size=128, n_merges=50)
print(f'vocab size: {len(tok.vocab)}')
print(f'n_merges:   {len(tok.merges)}')
sample = 'the cat is on the mat'
ids = tok.encode(sample)
print(f'encode({sample!r}) = {ids[:10]}... ({len(ids)} tokens)')
print(f'decode = {tok.decode(ids)[:60]!r}')

vocab size: 178
n_merges:   50
encode('the cat is on the mat') = [129, 137, 152, 134, 129, 138]... (6 tokens)
decode = 'the cat is on the mat'


In [4]:
# === Section 2: compression ratio ===
import numpy as np
for phrase in ['the cat', 'the cat ' * 5, 'machine learning', 'neural networks']:
    n_chars = len(phrase)
    n_tokens = len(tok.encode(phrase))
    ratio = n_chars / max(n_tokens, 1)
    print(f'  {phrase!r:<30}  chars={n_chars:>3}  tokens={n_tokens:>3}  ratio={ratio:.1f}')

  'the cat'                       chars=  7  tokens=  2  ratio=3.5
  'the cat the cat the cat the cat the cat '  chars= 40  tokens= 10  ratio=4.0
  'machine learning'              chars= 16  tokens=  7  ratio=2.3
  'neural networks'               chars= 15  tokens= 10  ratio=1.5


In [5]:
# === Section 3: mock LLM and cost tracker ===
model = _L8.MockLLM(model_id='mock-1', default_response='Hello, world!')
client = _L8.LLMClient(tokenizer=tok, model=model)
for q in _L8.SAMPLE_QUESTIONS:
    msgs = _L8.SAMPLE_TEMPLATE.render(question=q)
    out = client.complete(msgs)
print(f'total calls:      {client.call_count}')
print(f'total input tok:  {client.tracker.total_input_tokens()}')
print(f'total output tok: {client.tracker.total_output_tokens()}')
print(f'total cost USD:   ${client.tracker.total_cost_usd():.6f}')
print(f'mean latency ms:  {client.tracker.mean_latency_ms():.2f}')
print(f'p95 latency ms:   {client.tracker.p95_latency_ms():.2f}')

total calls:      5
total input tok:  403
total output tok: 234
total cost USD:   $0.000201
mean latency ms:  0.01
p95 latency ms:   0.01


In [6]:
# === Section 4: prompt template rendering ===
msgs = _L8.SAMPLE_TEMPLATE.render(question='What is BPE?')
print(f'role: {msgs[0]["role"]}')
print(f'content[:80]: {msgs[0]["content"][:80]!r}')
print(f'role: {msgs[1]["role"]}')
print(f'content: {msgs[1]["content"]!r}')

role: system
content[:80]: 'You are a helpful assistant.  Always answer concisely.'
role: user
content: 'Answer the following: What is BPE?'


In [7]:
# === Section 5: structured output parser ===
raw = '```json\n{"answer": "BPE is a subword tokenizer.", "confidence": 0.95}\n```'
parsed = _L8.parse_json_output(raw)
print(f'parsed: {parsed}')
print(f'type:   {type(parsed).__name__}')

parsed: {'answer': 'BPE is a subword tokenizer.', 'confidence': 0.95}
type:   dict


In [8]:
# === Section 6: cost projection at scale ===
monthly = client.tracker.total_cost_usd() / client.call_count * 1_000_000
annual = monthly * 12
print(f'per call:               ${client.tracker.total_cost_usd() / client.call_count:.6f}')
print(f'monthly (1M calls):     ${monthly:,.2f}')
print(f'annual:                 ${annual:,.2f}')

per call:               $0.000040
monthly (1M calls):     $40.17
annual:                 $482.04


In [9]:
# === Section 7: usage record detail ===
print(f'{"call_id":<10} {"in_tok":>7} {"out_tok":>7} {"latency_ms":>11} {"cost_usd":>10}')
for r in client.tracker.records:
    print(f'{r.call_id:<10} {r.input_tokens:>7} {r.output_tokens:>7} {r.latency_ms:>11.3f} {r.cost_usd:>10.6f}')

call_id     in_tok out_tok  latency_ms   cost_usd
call-0001       80      47       0.013   0.000040
call-0002       77      47       0.011   0.000040
call-0003       78      47       0.013   0.000040
call-0004       84      47       0.012   0.000041
call-0005       84      46       0.010   0.000040


## Architecture (Mermaid)

```mermaid
flowchart LR
    A[user input] --> T[PromptTemplate<br/>role: system / user]
    T --> TOK[Tokenizer<br/>encode]
    TOK --> M[LLM call<br/>stream / batch]
    M --> OUT[response text]
    OUT --> P[Output parser<br/>JSON / regex]
    P --> R[structured result]
    M --> LOG[Cost + latency log]
    LOG --> DB[(usage table)]
    LOG --> OBS[observability<br/>metrics / traces]
```

**Read as:** Template, tokenize, call, parse, log. Every LLM app has these five pieces; the differences are in the details.

## Reflect (10 min)

1. Why are tokens the unit of cost, not characters?
2. Walk through BPE in one sentence. Why does it compress repeated phrases?
3. The cost projection shows $20/month at 1M mock calls. What would it be for gpt-4o-mini at 1M real calls? gpt-4o?
4. Why is JSON mode the standard for structured output? What goes wrong without it?
5. (Stretch) Add a `cost_ceiling(client, ceiling_usd)` that raises if `tracker.total_cost_usd() > ceiling_usd`.

---

**Read next:**
- `lessons/m09-rag-e2e/L9-rag-end-to-end.ipynb` -- the full RAG pipeline
- `lessons/m10-evaluation/L10-evaluation-ragas.ipynb` -- RAGAS eval framework
- `SYLLABUS.md` -- the 18 modules and 21-day plan